# 📘 Colab Notebook: Evaluate LLM Responses from MongoDB Using Llumo

## 📝 Notebook Overview
This notebook demonstrates how to load your existing LLM interaction logs from a MongoDB collection, format them, and then evaluate the model's responses using Llumo’s powerful input-level metrics to ensure quality and safety.

### ✨ Metrics included:

- 🎯 Response Correctness
- 🧩 Response Completeness
- 🧠 Response Bias
- ☣️ Response Harmfulness
- ▶ Hallucination
- 🛠️ Context Utilization
  
---

## 🚀 What you will do in this notebook:
- 🍃 Connect to your MongoDB database and load a collection of interaction logs.  
- 🔄 Format the raw data from MongoDB documents into the standardized structure required by Llumo.
- 🤖 Evaluate the model's output for correctness, completeness, bias, harmfulness, and more.
- 📊 View the detailed evaluation results in a structured table.  
---

### **⚙️ 1. Install Dependencies**
First, we'll install the necessary Python libraries. `llumo` is the official SDK for the Llumo platform, `pymongo` is the standard driver for working with MongoDB, and `pandas` is used for data manipulation.

In [ ]:
# The '[srv]' is important for connecting to modern MongoDB Atlas instances.
!pip install llumo pymongo[srv] pandas -q

### **📚 2. Import Required Libraries**

In [ ]:
import os
import pandas as pd
import json
import getpass
from llumo import LlumoClient
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

### **🔑 3. Configure API Key & MongoDB Connection Details**

To use Llumo and access your MongoDB database, you need to set up the appropriate credentials.

1.  **Llumo API Key**: You can get your key from the [Llumo Dashboard](https://llumo.ai/dashboard).
2.  **MongoDB Connection String**: You'll need your database's connection string (URI). For MongoDB Atlas, you can find this in the "Connect" section of your cluster. 

In [ ]:
# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = getpass.getpass("Enter Your Llumo API Key: ")
llumo_key = os.getenv("LLUMO_API_KEY")

# ⚙️ Set your MongoDB connection details
mongo_connection_string = getpass.getpass("Enter Your MongoDB Connection String: ")
db_name = input("Enter your Database Name (e.g., 'llm_logs_db'): ")
collection_name = input("Enter your Collection Name (e.g., 'chat_sessions'): ")

### **🍃 4. Read Data from MongoDB**

This step connects to your MongoDB cluster, selects the specified database and collection, and fetches the documents. We will query the collection and load the documents into a list for processing.

**Note**: We are limiting the query to the first 100 documents for this example. You can adjust or remove the `.limit(100)` part to fetch more data.

In [ ]:
raw_logs = []
try:
    # Establish a connection to the MongoDB server
    print("Connecting to MongoDB...")
    client = MongoClient(mongo_connection_string)
    
    # The ismaster command is cheap and does not require auth.
    client.admin.command('ismaster')
    print("MongoDB connection successful.")
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]
    
    # Fetch documents from the collection (limited to 100 for this example)
    print(f"Fetching documents from '{collection_name}'...")
    # The '_id' field from MongoDB is not JSON serializable, so we exclude it.
    raw_logs = list(collection.find({}, {'_id': 0}).limit(100))
    print(f"Successfully loaded {len(raw_logs)} records from MongoDB.")

except ConnectionFailure as e:
    print(f"Could not connect to MongoDB: {e}")
    print("Please check your connection string and ensure your IP is whitelisted.")
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    # It's important to close the connection
    if 'client' in locals() and client:
        client.close()
        print("MongoDB connection closed.")

# Preview the first raw log to understand its structure
if raw_logs:
    print("\nSample raw log from MongoDB:")
    print(json.dumps(raw_logs[0], indent=2))

### **🔄 5. Format Data for Llumo Evaluation**
Llumo's `evaluateMultiple` function expects a list of dictionaries, where each dictionary contains specific keys like `query`, `context`, and `output`. 

The function below converts our raw logs from MongoDB into this standardized format. **You must adjust the key mappings** inside the function to match the field names in your own MongoDB documents.

In [ ]:
def convert_to_llumo_format(logs):
  """
  Converts a list of raw log dictionaries from MongoDB into the format required by Llumo.
  
  Args:
    logs (list): A list of dictionaries, where each dictionary is a document from MongoDB.
    
  Returns:
    list: A list of formatted dictionaries for Llumo evaluation.
  """
  formatted_data = []
  for log in logs:
    # ➡️ TODO: Adjust these key names to match your MongoDB document structure.
    # For example, if your user's prompt is stored in a field called 'question',
    # change 'userPrompt' to 'question'.
    formatted_dict = {
        'query': log.get('userPrompt', ''),        # Map your field for the user's question/prompt
        'context': log.get('retrievedContext', ''),  # Map your field for the retrieved context
        'output': log.get('modelResponse', ''),     # Map your field for the model's generated response
        # Optional: Map the ground truth field if you have one
        'ground_truth': log.get('referenceAnswer', None) 
    }
    formatted_data.append(formatted_dict)
  return formatted_data

# Process the loaded logs
if raw_logs:
    evaluation_data = convert_to_llumo_format(raw_logs)
    
    # Preview the first formatted item to verify the mapping
    print("Sample log after formatting for Llumo:")
    print(json.dumps(evaluation_data[0], indent=2))
else:
    evaluation_data = []
    print("No data to format.")

### **🤖 6. Initialize Llumo Client and Evaluate Responses**

Now we're ready for the evaluation. We will initialize the `LlumoClient` and call the `evaluateMultiple` function with our newly formatted data.

We pass our data and select the KPIs we want to measure:
- 🎯 **Response Correctness**: Is the answer factually accurate based on the context?
- 🧩 **Response Completeness**: Does the answer fully address the user's query?
- 🧠 **Response Bias**: Is the response free from demographic or social biases?
- ☣️ **Harmfulness**: Does the response contain toxic, hateful, or unsafe content?
- 🛠️ **Context Utilization**: How well does the answer use the provided context?
- ▶ **Hallucination**: Does the answer invent information not present in the context?

In [ ]:
evalDf = pd.DataFrame()

if evaluation_data and llumo_key:
    # Initialize the LlumoClient with your API key
    client = LlumoClient(api_key = llumo_key)

    # Call the evaluation function
    print("Starting evaluation with Llumo...")
    evalDf = client.evaluateMultiple(
      data = evaluation_data,  # The formatted data from the previous step
      evals = ["Response Completeness", "Response Correctness", "Response Bias", "Context Utilization", "Hallucination"], # Selected evaluation KPIs
      getDataFrame = True # Return result as a pandas DataFrame
    )
    print("Evaluation complete!")
else:
    print("Skipping evaluation. Ensure data was loaded from MongoDB and the Llumo API key is set.")

### **📊 7. View Evaluation Results**
The results are returned in a pandas DataFrame, providing a detailed breakdown of each metric for every data point. This allows for easy analysis, sorting, and filtering to identify problematic responses and gain insights into your model's performance.

In [ ]:
# Display the full evaluation results table
if not evalDf.empty:
    # Configure pandas to display wide columns for better readability
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 80)
    display(evalDf)
else:
    print("No evaluation results to display.")